# Published scores documented in clinical notes (regex, exploratory)

Feasibility test: how often do clinicians write the published prognostic scores (or their
missing inputs) directly in the chart? `build_published_scores` computes the scores from
structured data only; this notebook checks whether progress notes could fill the gaps:

- **Scores as documented**: IPI (incl. R-IPI / NCCN-IPI), MELD (incl. MELD-Na / MELD 3.0),
  ALBI (grade or score), CAPRA (incl. CAPRA-S), mGPS / GPS, LIPI, RMH, IMDC (Heng), MSKCC (Motzer).
- **Performance status**: ECOG and KPS, the item dropped from the ECOG-free IPI/IMDC/MSKCC.

For each score's eligible population (from `published_scores_df.csv.gz`, treatment anchor),
it reports the share of patients with a mention in a note dated 0-`LOOKBACK_DAYS` days before
the anchor (the same no-leakage rule as the lab window), any time before the anchor, and any
time at all. It also compares note values with the structured scores where both exist.

Regex output is noisy. Read the snippet samples before trusting a coverage number. Known traps:
`ipi` is also shorthand for ipilimumab (dose mentions like `ipi 1 mg/kg` are dropped; coverage is
only counted inside the aggressive-NHL population), `MSKCC` is also the hospital and `Heng` a
surname (risk-group patterns require the group word next to the name), and ECOG ranges like
`ECOG 1-2` keep both ends (`value`, `value_hi`).

Read-only: nothing here feeds the builder or the figures. Optional output (`SAVE_OUTPUTS`):
`FEATURE_PATH/published_scores_note_mentions.csv.gz` (one row per match, with a text
snippet, so it contains PHI and stays on the cluster) and
`FEATURE_PATH/published_scores_note_coverage.csv.gz`.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import polars as pl


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from anchors import date_col
from config import CLINICAL_NOTES_PATH, FEATURE_PATH, SURV_PATH
from pipelines.preprocessing import profile_sources as ps

ANCHOR = "treatment"
LOOKBACK_DAYS = 90
NOTE_TYPES = ("PROGRESS_NOTES",)  # add "PATHOLOGY_NOTES" / "IMAGING_NOTES" if wanted
SAVE_OUTPUTS = True
N_SNIPPETS = 8

pl.Config.set_tbl_rows(60)
pl.Config.set_fmt_str_lengths(200)
print(f"repo root: {REPO_ROOT}")

## Patterns

One entry per (mention, measure, regex). Every regex has a named `value` group; `variant`
(e.g. `R-IPI` vs `IPI`), `value_hi` (upper end of an ECOG range) and `unit` (used only to drop
ipilimumab doses) are optional. Matching is case-insensitive. Polars uses the Rust regex
engine, so there are no look-arounds; exclusions are applied after extraction instead.
`VALID_RANGE` drops values outside the score's possible range (e.g. `MELD 3` is not a MELD).

In [ ]:
_SEP = r"\s*(?:of|is|was|=|:)?\s*"
_GROUP = r"(?P<value>favou?rable|good|intermediate|poor)"

MENTION_PATTERNS: list[tuple[str, str, str]] = [
    ("ipi", "points",
     r"\b(?P<variant>R-?IPI|NCCN-?IPI|IPI)\b(?:\s+score)?" + _SEP + r"(?P<value>[0-8])\b(?P<unit>\s*mg)?"),
    ("meld", "continuous",
     r"\b(?P<variant>MELD(?:\s*3\.0)?(?:[- ]?Na)?)\b(?:\s+score)?" + _SEP + r"(?P<value>\d{1,2})\b"),
    ("albi", "grade",
     r"\bALBI\b(?:\s+score)?" + _SEP + r"(?:grade|gr\.?)\s*(?P<value>[123])\b"),
    ("albi", "continuous",
     r"\bALBI\b(?:\s+score)?" + _SEP + r"(?P<value>[-−–]\s?\d\.\d+)"),
    ("capra", "points",
     r"\b(?P<variant>CAPRA(?:-S)?)\b(?:\s+score)?" + _SEP + r"(?P<value>\d{1,2})\b"),
    ("mgps", "points",
     r"\b(?P<variant>m?GPS|(?:modified\s+)?Glasgow\s+prognostic\s+score)\b(?:\s+score)?" + _SEP + r"(?P<value>[0-2])\b"),
    ("lipi", "points",
     r"\bLIPI\b(?:\s+score)?" + _SEP + r"(?P<value>[0-2])\b"),
    ("lipi", "group",
     r"\bLIPI\b(?:\s+(?:score|group|category|risk))?\s*(?:of|is|was|=|:|-)?\s*" + _GROUP + r"\b"),
    ("lipi", "group",
     _GROUP + r"[- ](?:risk\s+)?(?:by\s+|per\s+)?LIPI\b"),
    ("rmh", "points",
     r"\b(?P<variant>RMH|Royal\s+Marsden(?:\s+Hospital)?)\s+(?:prognostic\s+)?score" + _SEP + r"(?P<value>[0-3])\b"),
    ("imdc", "group",
     r"\b(?P<variant>IMDC|Heng)\b(?:\s+(?:risk|prognostic))?(?:\s+(?:score|criteria|group|category|model|classification))?"
     r"\s*(?:of|is|was|=|:|-)?\s*" + _GROUP + r"\b"),
    ("imdc", "group",
     _GROUP + r"[- ]risk\b[^.\n]{0,25}?\b(?:by|per|according\s+to|based\s+on)\s+(?:the\s+)?(?P<variant>IMDC|Heng)\b"),
    ("imdc", "points",
     r"\b(?P<variant>IMDC)\b(?:\s+(?:risk\s+)?(?:score|criteria|risk\s+factors?))?" + _SEP + r"(?P<value>[0-6])\b"),
    ("imdc", "points",
     r"\b(?P<variant>Heng)\s+(?:risk\s+)?(?:score|criteria|risk\s+factors?)" + _SEP + r"(?P<value>[0-6])\b"),
    ("mskcc", "group",
     r"\b(?P<variant>MSKCC(?:/Motzer)?|Motzer)\b(?:\s+(?:risk|prognostic))?(?:\s+(?:score|criteria|group|category|model|classification))?"
     r"\s*(?:of|is|was|=|:|-)?\s*" + _GROUP + r"\b"),
    ("mskcc", "group",
     _GROUP + r"[- ]risk\b[^.\n]{0,25}?\b(?:by|per|according\s+to|based\s+on)\s+(?:the\s+)?(?P<variant>MSKCC|Motzer)\b"),
    ("mskcc", "points",
     r"\b(?P<variant>MSKCC(?:/Motzer)?|Motzer)\s+(?:risk\s+)?(?:score|criteria|risk\s+factors?)" + _SEP + r"(?P<value>[0-5])\b"),
    ("ecog", "points",
     r"\bECOG\b\)?(?:\s+(?:PS|performance\s+(?:status|score)))?\)?\s*(?:of|is|was|=|:|-)?\s*"
     r"(?P<value>[0-4])(?:\s*(?:-|to|/)\s*(?P<value_hi>[0-4]))?\b"),
    ("kps", "points",
     r"\b(?P<variant>KPS|Karnofsky(?:\s+performance\s+(?:status|scale|score))?)\b" + _SEP + r"(?P<value>100|[1-9]0)\b"),
]

VALID_RANGE = {
    ("ipi", "points"): (0, 8), ("meld", "continuous"): (6, 40), ("albi", "grade"): (1, 3),
    ("albi", "continuous"): (-5, 0), ("capra", "points"): (0, 12), ("mgps", "points"): (0, 2),
    ("lipi", "points"): (0, 2), ("rmh", "points"): (0, 3), ("imdc", "points"): (0, 6),
    ("mskcc", "points"): (0, 5), ("ecog", "points"): (0, 4), ("kps", "points"): (0, 100),
}

KEYWORD_RE = (
    r"(?i)\b(?:IPI|MELD|ALBI|CAPRA|m?GPS|Glasgow|LIPI|RMH|Marsden|IMDC|Heng|MSKCC|Motzer|ECOG|KPS|Karnofsky)\b"
)
_COMPILED = [f"(?i){pattern}" for _, _, pattern in MENTION_PATTERNS]
_SNIPPET = [f"(?i)([^\\n]{{0,80}}(?:{pattern})[^\\n]{{0,80}})" for _, _, pattern in MENTION_PATTERNS]
_OPTIONAL_GROUPS = ("variant", "value_hi", "unit")


def match_columns(text: pl.Expr) -> list[pl.Expr]:
    """Per-note list of matches plus one context snippet, for every pattern."""
    return [
        expr
        for i in range(len(MENTION_PATTERNS))
        for expr in (
            text.str.extract_all(_COMPILED[i]).alias(f"_m{i}"),
            text.str.extract(_SNIPPET[i], 1).alias(f"_s{i}"),
        )
    ]


def parse_matches(wide: pl.DataFrame, id_cols: list[str]) -> pl.DataFrame:
    """Wide per-note match lists -> one row per (note, match) with a parsed,
    normalized, range-checked value."""
    parts = []
    for i, (mention, measure, _) in enumerate(MENTION_PATTERNS):
        long = (
            wide.select(*id_cols, pl.col(f"_m{i}").alias("match"), pl.col(f"_s{i}").alias("snippet"))
            .explode("match", empty_as_null=True)
            .drop_nulls("match")
        )
        if long.is_empty():
            continue
        groups = long.with_columns(pl.col("match").str.extract_groups(_COMPILED[i]).alias("_g")).unnest("_g")
        parts.append(groups.select(
            *id_cols,
            pl.lit(mention).alias("mention"),
            pl.lit(measure).alias("measure"),
            pl.lit(i).alias("pattern_idx"),
            *[
                (pl.col(g) if g in groups.columns else pl.lit(None, dtype=pl.String)).alias(g)
                for g in _OPTIONAL_GROUPS
            ],
            pl.col("value"),
            "match",
            "snippet",
        ))
    if not parts:
        return pl.DataFrame()
    out = pl.concat(parts, how="diagonal_relaxed")
    value = (
        pl.col("value").str.to_lowercase().str.replace_all(r"[−–]", "-").str.replace_all(r"\s", "")
        .str.replace("favourable", "favorable")
    )
    out = out.with_columns(
        value.alias("value"),
        pl.col("variant").str.to_uppercase().str.replace_all(r"\s+", " "),
    ).with_columns(pl.col("value").cast(pl.Float64, strict=False).alias("value_num"))
    ranges = pl.DataFrame(
        [{"mention": k[0], "measure": k[1], "_lo": float(v[0]), "_hi": float(v[1])} for k, v in VALID_RANGE.items()]
    )
    out = out.join(ranges, on=["mention", "measure"], how="left")
    in_range = pl.col("value_num").is_null() | pl.col("value_num").is_between(pl.col("_lo"), pl.col("_hi"))
    is_dose = pl.col("unit").is_not_null()
    return out.filter(in_range & ~is_dose).drop("_lo", "_hi", "unit")

## Pattern self-test (runs without data)

Each case is `(text, mention, measure, expected value)`; `None` means the text must produce
no match for that mention. Add real failure cases here as the snippet review finds them.

In [ ]:
SELF_TEST = [
    ("IPI score of 3, high-intermediate", "ipi", "points", "3"),
    ("R-IPI: 2", "ipi", "points", "2"),
    ("NCCN-IPI 5/8", "ipi", "points", "5"),
    ("started ipi 1 mg/kg + nivo 3 mg/kg", "ipi", "points", None),
    ("MELD-Na 18 today", "meld", "continuous", "18"),
    ("MELD 3.0 score of 22", "meld", "continuous", "22"),
    ("ALBI grade 2", "albi", "grade", "2"),
    ("ALBI score -2.41", "albi", "continuous", "-2.41"),
    ("CAPRA score 4 (intermediate)", "capra", "points", "4"),
    ("mGPS 1", "mgps", "points", "1"),
    ("modified Glasgow prognostic score of 2", "mgps", "points", "2"),
    ("LIPI intermediate", "lipi", "group", "intermediate"),
    ("Royal Marsden score 2", "rmh", "points", "2"),
    ("IMDC intermediate risk", "imdc", "group", "intermediate"),
    ("poor risk per IMDC criteria", "imdc", "group", "poor"),
    ("IMDC score 2", "imdc", "points", "2"),
    ("MSKCC favourable risk", "mskcc", "group", "favorable"),
    ("seen at MSKCC last year for a second opinion", "mskcc", "group", None),
    ("ECOG PS 1", "ecog", "points", "1"),
    ("ECOG 1-2", "ecog", "points", "1"),
    ("Heng criteria: 3", "imdc", "points", "3"),
    ("discussed with Dr. Heng 2 weeks ago", "imdc", "points", None),
    ("performance status (ECOG): 0", "ecog", "points", "0"),
    ("enrolled on ECOG-ACRIN E1609", "ecog", "points", None),
    ("KPS 80%", "kps", "points", "80"),
]

_test_notes = pl.DataFrame({"RPT_ID": [str(i) for i in range(len(SELF_TEST))], "RPT_TEXT": [t for t, *_ in SELF_TEST]})
_test_wide = _test_notes.select("RPT_ID", *match_columns(pl.col("RPT_TEXT")))
_test_long = parse_matches(_test_wide, ["RPT_ID"])

failures = []
for i, (text, mention, measure, expected) in enumerate(SELF_TEST):
    got = _test_long.filter(
        (pl.col("RPT_ID") == str(i)) & (pl.col("mention") == mention) & (pl.col("measure") == measure)
    )["value"].to_list()
    ok = (expected is None and not got) or (expected is not None and got[:1] == [expected])
    if not ok:
        failures.append((text, mention, measure, expected, got))

_range = _test_long.filter(pl.col("match").str.contains("1-2") & (pl.col("mention") == "ecog"))
if _range["value_hi"].to_list() != ["2"]:
    failures.append(("ECOG 1-2", "ecog", "value_hi", "2", _range["value_hi"].to_list()))

for failure in failures:
    print("FAIL:", failure)
assert not failures, f"{len(failures)} self-test case(s) failed"
print(f"all {len(SELF_TEST)} self-test cases passed")

## Cohort, anchor and eligibility

Eligibility comes from the builder's treatment-anchor output, so the populations match the
structured-score analysis exactly. If that file is missing, every cohort patient counts as
eligible for everything (coverage numbers are then cohort-wide).

In [ ]:
anchor_col = date_col(ANCHOR)
cohort = pl.read_parquet(os.path.join(SURV_PATH, "cohort_df.parquet"), columns=["DFCI_MRN", anchor_col])
cohort = cohort.with_columns(
    pl.col("DFCI_MRN").cast(pl.Int64, strict=False),
    pl.col(anchor_col).cast(pl.Date, strict=False).alias("_anchor_date"),
).select("DFCI_MRN", "_anchor_date")

SCORE_IDS = ("mgps", "rmh", "lipi", "albi", "meld", "capra_mod", "ipi_noecog", "imdc_noecog", "mskcc_noecog")
scores_path = os.path.join(FEATURE_PATH, "published_scores_df.csv.gz")
if os.path.exists(scores_path):
    structured = pl.read_csv(scores_path, schema_overrides={"DFCI_MRN": pl.Int64}, infer_schema_length=None)
else:
    print(f"{scores_path} not found; treating every cohort patient as eligible")
    structured = cohort.select("DFCI_MRN", *[pl.lit(True).alias(f"{s}__eligible") for s in SCORE_IDS])

# Which structured-score populations each note mention is counted against.
MENTION_POPULATIONS = {
    "mgps": ["mgps"], "rmh": ["rmh"], "lipi": ["lipi"], "albi": ["albi"], "meld": ["meld"],
    "capra": ["capra_mod"], "ipi": ["ipi_noecog"], "imdc": ["imdc_noecog"], "mskcc": ["mskcc_noecog"],
    "ecog": ["ipi_noecog", "imdc_noecog", "mskcc_noecog", "all"],
    "kps": ["ipi_noecog", "imdc_noecog", "mskcc_noecog", "all"],
}
populations = {"all": cohort.select("DFCI_MRN")}
for score_id in SCORE_IDS:
    populations[score_id] = structured.filter(pl.col(f"{score_id}__eligible").fill_null(False)).select("DFCI_MRN")
print(pl.DataFrame({"population": list(populations), "n": [p.height for p in populations.values()]}))

## Scan notes

Lazy and streaming: note text is never fully materialized. Notes are filtered to cohort
patients and to a cheap keyword prefilter before any extraction. `EVENT_DATE` is tz-aware UTC
(or a string on some releases) and is localized to a naive date, as in `build_cohort`.

In [ ]:
cohort_mrns = cohort["DFCI_MRN"].implode()
wide_parts = []
for note_type in NOTE_TYPES:
    path = os.path.join(CLINICAL_NOTES_PATH, f"{note_type}.parquet")
    lf = pl.scan_parquet(path)
    event_dt = pl.col(ps.EVENT_DATE)
    if lf.collect_schema()[ps.EVENT_DATE] == pl.String:
        event_dt = event_dt.str.to_datetime(time_zone="UTC", strict=False)
    wide = (
        lf.select(ps.MRN, ps.RPT_ID, ps.EVENT_DATE, ps.RPT_TEXT)
        .with_columns(pl.col(ps.MRN).cast(pl.Int64, strict=False))
        .filter(pl.col(ps.MRN).is_in(cohort_mrns) & pl.col(ps.RPT_TEXT).str.contains(KEYWORD_RE))
        .select(
            pl.col(ps.MRN).alias("DFCI_MRN"),
            pl.col(ps.RPT_ID).cast(pl.String).alias("RPT_ID"),
            event_dt.dt.replace_time_zone(None).dt.date().alias("note_date"),
            pl.lit(note_type).alias("note_type"),
            *match_columns(pl.col(ps.RPT_TEXT).cast(pl.String)),
        )
        .collect(engine="streaming")
    )
    print(f"{note_type}: {wide.height:,} keyword-matching cohort notes")
    wide_parts.append(wide)

wide_all = pl.concat(wide_parts, how="vertical_relaxed")
mentions = parse_matches(wide_all, ["DFCI_MRN", "RPT_ID", "note_date", "note_type"])
mentions = mentions.join(cohort, on="DFCI_MRN", how="inner").with_columns(
    (pl.col("_anchor_date") - pl.col("note_date")).dt.total_days().alias("days_before_anchor")
).drop("_anchor_date")
del wide_all, wide_parts
print(f"{mentions.height:,} parsed mentions")
print(mentions.group_by("mention", "measure").len().sort("mention", "measure"))

## Coverage by population

Share of each population with at least one parsed mention, in three windows:
`0-LOOKBACK_DAYS` days before the anchor (no leakage, usable as a score input),
any time before the anchor, and any time (upper bound on documentation).

In [ ]:
windows = {
    f"le{LOOKBACK_DAYS}d": pl.col("days_before_anchor").is_between(0, LOOKBACK_DAYS),
    "before_anchor": pl.col("days_before_anchor") >= 0,
    "any_time": pl.lit(True),
}
rows = []
for mention, pops in MENTION_POPULATIONS.items():
    mention_rows = mentions.filter(pl.col("mention") == mention)
    for pop in pops:
        pop_ids = populations[pop]
        row = {"mention": mention, "population": pop, "n_population": pop_ids.height}
        for label, window in windows.items():
            n = mention_rows.filter(window).join(pop_ids, on="DFCI_MRN", how="semi")["DFCI_MRN"].n_unique()
            row[f"n_{label}"] = n
            row[f"pct_{label}"] = round(100 * n / pop_ids.height, 1) if pop_ids.height else None
        rows.append(row)
coverage = pl.DataFrame(rows)
coverage

## Values and snippets for manual review

Value distribution per mention within the no-leakage window and population, then a random
sample of snippets per (mention, measure). Check the snippets before trusting coverage.

In [ ]:
in_window = windows[f"le{LOOKBACK_DAYS}d"]
for mention, pops in MENTION_POPULATIONS.items():
    rows_ = mentions.filter((pl.col("mention") == mention) & in_window).join(populations[pops[0]], on="DFCI_MRN", how="semi")
    if rows_.is_empty():
        print(f"--- {mention} ({pops[0]}): no mentions in window")
        continue
    print(f"--- {mention} ({pops[0]})")
    print(rows_.group_by("measure", "variant", "value").len().sort("measure", "len", descending=[False, True]).head(15))

In [ ]:
for (mention, measure), group in mentions.group_by("mention", "measure", maintain_order=True):
    sample = group.sample(n=min(N_SNIPPETS, group.height), seed=0)
    print(f"\n=== {mention} / {measure}: {group.height:,} matches ===")
    for snippet, value in sample.select("snippet", "value").iter_rows():
        print(f"  [{value}] ...{snippet}...")

## Agreement with the structured scores

For each patient, the latest in-window mention (per mention and measure) against the
builder's structured value. Structured IPI/IMDC/MSKCC are ECOG-free lower bounds, so a note
value at or above the structured points is consistent; mGPS/LIPI/RMH/ALBI/MELD should match
directly up to lab-timing differences.

In [ ]:
STRUCTURED_COLUMN = {
    "mgps": "mgps__points", "rmh": "rmh__points", "lipi": "lipi__points", "albi": "albi__continuous",
    "meld": "meld__continuous", "capra": "capra_mod__points", "ipi": "ipi_noecog__points",
    "imdc": "imdc_noecog__points", "mskcc": "mskcc_noecog__points",
}
latest = (
    mentions.filter(in_window)
    .sort("days_before_anchor")
    .group_by("DFCI_MRN", "mention", "measure", maintain_order=True)
    .first()
)
for mention, column in STRUCTURED_COLUMN.items():
    group_col = column.split("__")[0] + "__group"
    if column not in structured.columns:
        continue
    joined = latest.filter(pl.col("mention") == mention).join(
        structured.select(
            "DFCI_MRN",
            pl.col(column).cast(pl.Float64, strict=False).alias("structured"),
            pl.col(group_col).cast(pl.String).alias("structured_group"),
        ),
        on="DFCI_MRN", how="inner",
    ).filter(pl.col("structured").is_not_null())
    if joined.is_empty():
        print(f"--- {mention}: no patients with both a note value and a structured value")
        continue
    print(f"--- {mention}: {joined.height} patients with both")
    numeric = joined.filter(pl.col("value_num").is_not_null() & (pl.col("measure") != "grade"))
    if not numeric.is_empty():
        print(numeric.select(
            pl.len().alias("n"),
            (pl.col("value_num") == pl.col("structured").round(0)).mean().alias("frac_equal"),
            (pl.col("value_num") >= pl.col("structured").round(0)).mean().alias("frac_note_ge_structured"),
            pl.corr("value_num", "structured", method="spearman").alias("spearman"),
        ))
    print(joined.group_by("measure", "value", "structured_group").len().sort("measure", "value", "structured_group"))

## Save

In [ ]:
if SAVE_OUTPUTS:
    mentions_path = os.path.join(FEATURE_PATH, "published_scores_note_mentions.csv.gz")
    coverage_path = os.path.join(FEATURE_PATH, "published_scores_note_coverage.csv.gz")
    mentions.write_csv(mentions_path, compression="gzip")
    coverage.write_csv(coverage_path, compression="gzip")
    print(f"wrote {mentions_path}\nwrote {coverage_path}")